In [1]:
from torch.optim import AdamW
from transformers import get_scheduler
from transformers import AutoModelForCausalLM

# Load pretrained model from local.
local_model_path = './models/tiny-story-summarize-19M-73K'
pretrained_model = AutoModelForCausalLM.from_pretrained(local_model_path)

/Users/sauravprateek/Documents/saurav-codes/neural-nets-codelab/saurav-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 108/108 [00:00<00:00, 2281.28it/s, Materializing param=transformer.wte.weight]                       


In [2]:
from transformers import AutoTokenizer, TextStreamer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
tokenizer.pad_token = tokenizer.eos_token

In [3]:
def summarize_story(model, story_summary, logging=True):
    # Move model to evaluation mode
    model.eval()

    # Prompt with a typical TinyStories opening
    story_prompt = story_summary.split('\nSummary: ')[0].strip()
    prompt = story_prompt + '\nSummary: '

    if logging:
        print(prompt)

    inputs = tokenizer(prompt, return_tensors="pt")
    if logging:
        print(f'Shape of Input prompt: {inputs['input_ids'].shape}')

    # Generate
    output_tokens = pretrained_model.generate(
        inputs.input_ids,
        max_new_tokens=100, 
        do_sample=True,
        temperature=1, 
        top_k=50,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_output_tokens = output_tokens[0][inputs.input_ids.shape[1]:]
    decoded_output = tokenizer.decode(generated_output_tokens)
    return decoded_output

In [4]:
from datasets import load_from_disk

tokenized_dataset = load_from_disk('./datasets/tiny-story-summary-tokenized')

In [5]:
tokenized_dataset

Dataset({
    features: ['input_ids'],
    num_rows: 100000
})

In [8]:
inference_id = 9005
decoded_story = tokenizer.decode(tokenized_dataset[inference_id])
print(f'{decoded_story}\n\n')

generated_summary = summarize_story(pretrained_model, decoded_story, logging=False)

print(f'Summary: {generated_summary}')

Story: Once upon a time, there was a big, fat penguin named Puddles. Puddles loved to play with his friends on the ice. One day, Puddles saw a big block of ice and decided to cut it. He used his sharp beak to cut the ice into small pieces. Puddles and his friends had fun sliding on the ice pieces. They laughed and played until it was time to go home. Puddles went to bed that night feeling happy and proud that he was able to cut the ice. Summary: Puddles the penguin cuts a block of ice into small pieces to play with his friends on the ice. 
Summary: Puddles the penguin cuts a block of ice into small pieces to play with his friends on the ice.<|endoftext|>


Summary: 
Summary: Puddles the penguin, a big fat penguin, cuts a block into small pieces to play with his friends on the ice.<|endoftext|>


In [ ]:
prompt = '''
Story: Once upon a time, there was a big, fat penguin named Puddles. Puddles loved to play with his friends on the ice. One day, Puddles saw a big block of ice and decided to cut it. He used his sharp beak to cut the ice into small pieces. Puddles and his friends had fun sliding on the ice pieces. They laughed and played until it was time to go home. Puddles went to bed that night feeling happy and proud that he was able to cut the ice.

Summary: 
'''

inputs = tokenizer(prompt, return_tensors="pt")

# Generate
output_tokens = pretrained_model.generate(
    inputs.input_ids,
    max_new_tokens=100, 
    do_sample=True,
    temperature=1, 
    top_k=50,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

In [12]:
print(tokenizer.decode(output_tokens)[0])


Story: Once upon a time, there was a big, fat penguin named Puddles. Puddles loved to play with his friends on the ice. One day, Puddles saw a big block of ice and decided to cut it. He used his sharp beak to cut the ice into small pieces. Puddles and his friends had fun sliding on the ice pieces. They laughed and played until it was time to go home. Puddles went to bed that night feeling happy and proud that he was able to cut the ice.

Summary: 
Summary: Puddles, a big stuffed penguin, accidentally cuts a big block of ice and has fun playing with his friends.<|endoftext|>


In [ ]:
# pretrained_model.push_to_hub('tiny-stories-19M-instruct')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 38.52it/s]
Processing Files (1 / 1): 100%|██████████| 77.2MB / 77.2MB, 1.98MB/s  
New Data Upload: 100%|██████████| 77.2MB / 77.2MB, 1.98MB/s  


CommitInfo(commit_url='https://huggingface.co/SauravP97/tiny-stories-19M-instruct/commit/6029147ba2d41d631eb379a3cb0aa6846c674786', commit_message='Upload GPTNeoForCausalLM', commit_description='', oid='6029147ba2d41d631eb379a3cb0aa6846c674786', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SauravP97/tiny-stories-19M-instruct', endpoint='https://huggingface.co', repo_type='model', repo_id='SauravP97/tiny-stories-19M-instruct'), pr_revision=None, pr_num=None)